# EV Charging Network Monitor — eco-movement & Tesla

This notebook reads the dynamic status databases for **eco-movement** and
**Tesla** and visualizes the number of charging points actively in use
over time.

The collector stores status *transitions* (change-detection) between
periodic **SNAPSHOT** deliveries.  Pure delta tracking drifts over time
because some transitions are missed between polls.  We correct this by:

1. Using each **SNAPSHOT** as a ground-truth anchor for the full
   system state.
2. Replaying deltas between consecutive SNAPSHOTs to capture the
   detailed *shape* of usage fluctuations.
3. Applying a **proportional correction** so that the replayed curve
   matches the ground truth at both SNAPSHOT endpoints—preserving
   the delta-driven shape while eliminating accumulated drift.

## Download Current Data from Server

The collectors run continuously on a remote server.
Run the cell below to **sync the latest SQLite databases** to your
local `data/` folder.  After syncing you can work entirely offline.

> **Note:** Update the `SERVER` variable with your own deployment details.


In [ ]:
import subprocess, sys, os, shutil
from pathlib import Path

# ── Configure these for your deployment ───────────────────────────────
SERVER = os.environ.get("EV_PULSE_SERVER", "user@your-server-ip")
SSH_KEY = Path(os.environ.get("EV_PULSE_SSH_KEY", Path.home() / ".ssh" / "id_ed25519"))
REMOTE_DIR = os.environ.get("EV_PULSE_REMOTE_DIR", "/home/user/ev-pulse/data")
LOCAL_DIR = Path("data")
LOCAL_DIR.mkdir(exist_ok=True)

FILES = [f"{p}_{k}.sqlite" for p in ("eco", "tesla") for k in ("static", "dynamic")]

# Step 1: Create consistent backups on the server using Python's
# sqlite3.backup() API. This is safe even while collectors are writing.
backup_script = (
    "import sqlite3, pathlib, sys; "
    "[("
    "  src := sqlite3.connect(f'data/{f}'),"
    "  dst := sqlite3.connect(f'data/{f}.bak'),"
    "  src.backup(dst),"
    "  dst.close(),"
    "  src.close(),"
    "  print(f'{f}: ok')"
    f") for f in {FILES!r} if pathlib.Path(f'data/{{f}}').exists()]"
)

print("Creating consistent snapshots on server …")
r = subprocess.run(
    ["ssh", "-i", str(SSH_KEY), SERVER,
     f"cd {REMOTE_DIR}/.. && python3 -c \"{backup_script}\""],
    capture_output=True, text=True,
)
print(r.stdout.strip() if r.stdout.strip() else f"stderr: {r.stderr.strip()}")
print()

# Step 2: Download the .bak copies
for fname in FILES:
    remote = f"{SERVER}:{REMOTE_DIR}/{fname}.bak"
    local_tmp = LOCAL_DIR / f"{fname}.tmp"
    local_final = LOCAL_DIR / fname
    print(f"Syncing {fname} … ", end="", flush=True)
    result = subprocess.run(
        ["scp", "-i", str(SSH_KEY), remote, str(local_tmp)],
        capture_output=True, text=True,
    )
    if result.returncode == 0:
        shutil.move(str(local_tmp), str(local_final))
        size_mb = local_final.stat().st_size / 1024 / 1024
        print(f"{size_mb:.1f} MB")
    else:
        if local_tmp.exists():
            os.remove(str(local_tmp))
        print(f"FAILED: {result.stderr.strip()}")

# Step 3: Clean up .bak files on server
cleanup = "; ".join(f"rm -f {REMOTE_DIR}/{f}.bak" for f in FILES)
subprocess.run(["ssh", "-i", str(SSH_KEY), SERVER, cleanup],
               capture_output=True, text=True)
print("\nDone.")

## Current State of the Charging Network

The **current state** is the result of replaying the last **SNAPSHOT**
(a full dump of every point's status) plus all subsequent **DELTA**
updates on top of it.  This gives the most recent known status of
every charging point in the system — entirely from local data.


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from collections import Counter

PROVIDERS = {
    "eco_movement": {"slug": "eco",   "in_use": "charging", "label": "eco-movement"},
    "tesla":        {"slug": "tesla", "in_use": "occupied", "label": "Tesla"},
}


def current_state(slug: str) -> pd.DataFrame:
    """Return the last known status of every charging point.

    This is equivalent to the last SNAPSHOT + all subsequent DELTAs:
    for each point_id we take the row with the highest id (= most recent
    update), giving us the current status.
    """
    db = Path("data") / f"{slug}_dynamic.sqlite"
    with sqlite3.connect(db) as conn:
        return pd.read_sql_query(
            """
            SELECT h.point_id, h.status, h.collected_at_utc
            FROM point_status_history h
            INNER JOIN (
                SELECT point_id, MAX(id) AS max_id
                FROM point_status_history
                GROUP BY point_id
            ) latest ON h.id = latest.max_id
            """,
            conn,
        )


def static_info(slug: str) -> pd.DataFrame:
    """Load static metadata (coordinates, name, address, …)."""
    db = Path("data") / f"{slug}_static.sqlite"
    with sqlite3.connect(db) as conn:
        return pd.read_sql_query("SELECT * FROM charging_points", conn)


# ── Compute and display current state per provider ────────────────────
for name, cfg in PROVIDERS.items():
    slug = cfg["slug"]
    state = current_state(slug)
    static = static_info(slug)

    state["collected_at_utc"] = pd.to_datetime(state["collected_at_utc"])
    latest_update = state["collected_at_utc"].max()

    dist = Counter(state["status"])
    total = len(state)
    in_use = dist.get(cfg["in_use"], 0)

    print(f"\n{'═' * 55}")
    print(f"  {cfg['label']}  —  {total:,} points tracked")
    print(f"  Latest update: {latest_update:%Y-%m-%d %H:%M:%S} UTC")
    print(f"  Static metadata: {len(static):,} points")
    print(f"{'─' * 55}")
    for st, cnt in dist.most_common():
        bar = "█" * int(cnt / total * 40)
        print(f"  {st or '(none)':20s}  {cnt:>6,}  ({cnt/total*100:5.1f}%)  {bar}")
    print(f"{'═' * 55}")


In [ ]:
import matplotlib.pyplot as plt

# ── Map: currently in-use charging points ─────────────────────────────
LAT_MIN, LAT_MAX = 47.2, 55.1
LON_MIN, LON_MAX = 5.8, 15.1

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, (name, cfg) in zip(axes, PROVIDERS.items()):
    slug = cfg["slug"]
    state = current_state(slug)
    static = static_info(slug)

    # Merge to get coordinates for in-use points
    in_use_ids = set(state.loc[state["status"] == cfg["in_use"], "point_id"])
    geo = static[static["point_id"].isin(in_use_ids)].dropna(subset=["latitude", "longitude"])

    if not geo.empty:
        ax.hexbin(
            geo["longitude"], geo["latitude"],
            gridsize=40, cmap="YlOrRd", mincnt=1,
            extent=[LON_MIN, LON_MAX, LAT_MIN, LAT_MAX],
        )
    ax.set_xlim(LON_MIN, LON_MAX)
    ax.set_ylim(LAT_MIN, LAT_MAX)
    ax.set_aspect(1.4)
    ax.set_title(f"{cfg['label']}  —  {len(in_use_ids):,} in use now", fontsize=12)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, alpha=0.2)

fig.suptitle("Current Charging Point Usage", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()


## Estimated Power Draw Over Time (MW)

Each charging point has a **rated power** in the static metadata
(`point_power_w` for eco-movement, `point_power_kw` for Tesla).
By replaying the dynamic history we know *which* points are in use at
each moment; summing their rated power gives an upper-bound estimate
of the instantaneous grid load from EV charging.

> **Note:** This is the *nameplate* power, not actual metered consumption.
> Real draw depends on vehicle SOC, cable limits, and load management.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

# ── Provider config with power column names ───────────────────────────
POWER_CFG = {
    "eco": {
        "in_use": "charging",
        "power_col": "point_power_w",   # stored in watts
        "to_mw": 1e-6,                  # W → MW
        "color": "tab:blue",
        "label": "eco-movement",
    },
    "tesla": {
        "in_use": "occupied",
        "power_col": "point_power_kw",  # stored in kilowatts
        "to_mw": 1e-3,                  # kW → MW
        "color": "tab:red",
        "label": "Tesla",
    },
}


def load_power_timeline(slug: str) -> tuple[pd.DataFrame, list[pd.Timestamp]]:
    """Compute per-charger MW timeline with drift correction.

    Returns (timeline_df, snapshot_timestamps).

    Replays SNAPSHOT + DELTA history, then applies proportional
    correction between consecutive SNAPSHOTs so the curve matches
    ground truth at both endpoints while preserving delta-driven shape.
    """
    data_dir = LOCAL_DIR if "LOCAL_DIR" in globals() else Path("data")
    dyn_db = data_dir / f"{slug}_dynamic.sqlite"
    sta_db = data_dir / f"{slug}_static.sqlite"
    cfg = POWER_CFG[slug]
    empty = pd.DataFrame(columns=["time", "power_mw"]), []

    if not dyn_db.exists() or not sta_db.exists():
        return empty

    # ── Per-point rated power in MW ──────────────────────────────────
    with sqlite3.connect(sta_db) as conn:
        pdf = pd.read_sql_query(
            f"SELECT point_id, {cfg['power_col']} FROM charging_points "
            f"WHERE {cfg['power_col']} IS NOT NULL",
            conn,
        )
    power_map = dict(zip(pdf["point_id"], pdf[cfg["power_col"]] * cfg["to_mw"]))

    # ── Full event log ───────────────────────────────────────────────
    with sqlite3.connect(dyn_db) as conn:
        df = pd.read_sql_query(
            "SELECT h.point_id, h.status, h.collected_at_utc, "
            "       h.snapshot_id, s.delivery_type "
            "FROM point_status_history h "
            "JOIN snapshot_runs s ON h.snapshot_id = s.snapshot_id "
            "ORDER BY h.id",
            conn,
        )
    if df.empty:
        return empty

    df["collected_at_utc"] = pd.to_datetime(
        df["collected_at_utc"], format="ISO8601", utc=True, errors="coerce",
    )
    df = df.dropna(subset=["collected_at_utc"])
    if df.empty:
        return empty

    in_use = cfg["in_use"]

    # ── Pass 1: Raw replay ───────────────────────────────────────────
    state: dict[str, str] = {}   # point_id → current status
    running_mw = 0.0
    records: list[tuple] = []    # (timestamp, raw_power_mw)
    snapshot_indices: list[int] = []  # indices into records where SNAPSHOTs occur
    snapshot_times: list[pd.Timestamp] = []

    for (sid, ts), group in df.groupby(["snapshot_id", "collected_at_utc"], sort=False):
        if group["delivery_type"].iat[0] == "SNAPSHOT":
            # Full state reset — recompute from scratch
            state = dict(zip(group["point_id"], group["status"]))
            running_mw = sum(
                power_map.get(pid, 0.0)
                for pid, s in state.items()
                if s == in_use
            )
            snapshot_indices.append(len(records))
            snapshot_times.append(ts)
        else:
            # Delta — adjust running total incrementally
            for pid, new_status in zip(group["point_id"], group["status"]):
                old_status = state.get(pid)
                pw = power_map.get(pid, 0.0)
                if old_status == in_use and new_status != in_use:
                    running_mw -= pw
                elif old_status != in_use and new_status == in_use:
                    running_mw += pw
                state[pid] = new_status

        records.append((ts, running_mw))

    result = pd.DataFrame(records, columns=["time", "power_mw"]).set_index("time").sort_index()

    # ── Pass 2: Proportional drift correction ────────────────────────
    if len(snapshot_indices) >= 2:
        power_vals = result["power_mw"].values
        times = result.index

        for i in range(len(snapshot_indices) - 1):
            idx_a = snapshot_indices[i]
            idx_b = snapshot_indices[i + 1]

            if idx_b <= idx_a + 1:
                continue

            power_at_b = power_vals[idx_b]
            power_before_b = power_vals[idx_b - 1]
            jump = power_before_b - power_at_b

            if abs(jump) < 0.01:
                continue

            t_a = times[idx_a].timestamp()
            t_b = times[idx_b].timestamp()
            span = t_b - t_a
            if span <= 0:
                continue

            for j in range(idx_a + 1, idx_b):
                frac = (times[j].timestamp() - t_a) / span
                power_vals[j] -= jump * frac

    return result, snapshot_times


# ── Compute power timelines ──────────────────────────────────────────
power_series = {}
snapshot_ts = {}
for slug, cfg in POWER_CFG.items():
    ts, snaps = load_power_timeline(slug)
    if not ts.empty:
        power_series[slug] = ts
        snapshot_ts[slug] = snaps
        print(f"{cfg['label']:15s}: {len(ts):>7,} samples, "
              f"peak = {ts['power_mw'].max():.1f} MW, "
              f"current = {ts['power_mw'].iloc[-1]:.1f} MW, "
              f"SNAPSHOTs = {len(snaps)}")
    else:
        print(f"{cfg['label']:15s}: no data")

# ── Combined total ────────────────────────────────────────────────────
if len(power_series) == 2:
    eco_ts = power_series["eco"].resample("1min").last().ffill()
    tesla_ts = power_series["tesla"].resample("1min").last().ffill()
    combined = eco_ts.add(tesla_ts, fill_value=0)
    print(f"\n{'Combined':15s}: peak = {combined['power_mw'].max():.1f} MW, "
          f"current = {combined['power_mw'].iloc[-1]:.1f} MW")

In [ ]:
# ── Plot: Estimated Power Draw (MW) ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

# Collect all SNAPSHOT times across providers for vertical markers
all_snap_times = set()
for snaps in snapshot_ts.values():
    all_snap_times.update(snaps)
all_snap_times = sorted(all_snap_times)

# Top: stacked area
if len(power_series) == 2:
    eco_r = power_series["eco"].resample("1min").last().ffill()
    tesla_r = power_series["tesla"].resample("1min").last().ffill()
    idx = eco_r.index.union(tesla_r.index)
    eco_v = eco_r.reindex(idx).ffill().fillna(0)["power_mw"]
    tesla_v = tesla_r.reindex(idx).ffill().fillna(0)["power_mw"]

    ax1.fill_between(idx, 0, eco_v, alpha=0.4,
                     color="tab:blue", label="eco-movement")
    ax1.fill_between(idx, eco_v, eco_v + tesla_v, alpha=0.4,
                     color="tab:red", label="Tesla")
    ax1.plot(idx, eco_v + tesla_v, linewidth=0.6, color="black",
             alpha=0.5, label="Total")
else:
    for slug, ts in power_series.items():
        cfg = POWER_CFG[slug]
        ax1.fill_between(ts.index, ts["power_mw"], alpha=0.4,
                         color=cfg["color"], label=cfg["label"])

# SNAPSHOT markers on top plot
for snap_t in all_snap_times:
    ax1.axvline(snap_t, color="black", linewidth=0.8, linestyle="--", alpha=0.5)
if all_snap_times:
    ax1.axvline(all_snap_times[0], color="black", linewidth=0.8,
                linestyle="--", alpha=0.5, label="SNAPSHOT")

ax1.set_ylabel("Power Draw (MW)")
ax1.set_title("Estimated Power Drawn from In-Use Chargers")
ax1.legend(loc="upper left")
ax1.grid(True, alpha=0.3)
ax1.set_ylim(bottom=0)

# Bottom: per-provider lines
for slug, ts in power_series.items():
    cfg = POWER_CFG[slug]
    ax2.plot(ts.index, ts["power_mw"], linewidth=0.7,
             color=cfg["color"], label=cfg["label"])

# SNAPSHOT markers on bottom plot
for snap_t in all_snap_times:
    ax2.axvline(snap_t, color="black", linewidth=0.8, linestyle="--", alpha=0.5)

ax2.set_ylabel("MW")
ax2.set_xlabel("Time (UTC)")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(bottom=0)

ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d\n%H:%M"))
ax2.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.tight_layout()
plt.show()